# FiCo-ITR Tutorial: Unified Image-Text Retrieval Evaluation

This tutorial demonstrates FiCo-ITR's unified evaluation approach for image-text retrieval models.

[![Paper](https://img.shields.io/badge/paper-Springer-blue)](https://link.springer.com/article/10.1007/s13735-025-00368-6)
[![GitHub](https://img.shields.io/badge/github-FiCo--ITR-green)](https://github.com/MikelWL/FiCo-ITR)

## What we'll cover

1. **Basic concepts** - Instance vs Category retrieval
2. **Four main variations** - How different models format their outputs
3. **Complete evaluation** - Running all models from our demo

In [1]:
#@title ⚡ **Run this first** - Setup and download data { display-mode: "form" }
#@markdown Choose download option based on your needs

download_option = "demo" #@param ["demo", "manual", "skip"]

import os
print("Installing FiCo-ITR...")
!pip install -q fico_itr numpy gdown

if download_option == "demo":
    print("\n📦 Downloading demo subset (~200MB)...")
    print("This may take a minute depending on your connection.")

    # Download with progress bar
    !gdown --folder https://drive.google.com/drive/folders/1gNxrtjoRT1jRP0CSSmp7329o18VP_Gs4?usp=sharing -O /content/results_data/

    # Move files from subdirectory if needed
    print("\n📂 Organizing files...")
    !mv /content/results_data/demo_data/* /content/results_data/ 2>/dev/null || echo "Files already in correct location"
    !rmdir /content/results_data/demo_data 2>/dev/null || true

elif download_option == "manual":
    print("\n📋 Manual Download Instructions:")
    print("1. Download files from: https://drive.google.com/drive/folders/1QiLlhJdcGO_bTyPh7LnK7y4qN5EjGBQd")
    print("2. Upload to Colab using the sidebar file browser")
    print("3. Move files to /content/results_data/")

else:  # skip
    print("\n⏭️ Skipping download - assuming data is already available")

# Setup
!mkdir -p /content/results_data
os.chdir('/content')

# Check what's available
import glob
files = glob.glob('results_data/*.npy')
if files:
    models = {f.split('/')[-1].split('_')[0] for f in files
              if '_' in f and not f.split('/')[-1].startswith(('flickr', 'coco', 'mscoco'))}
    total_size = sum(os.path.getsize(f) for f in files) / (1024*1024)
    print(f"✅ Found {len(files)} files ({total_size:.0f}MB) | Models: {', '.join(sorted(models))}")
else:
    print("⚠️ No data files found. Check the download or try 'manual' option.")

Installing FiCo-ITR...

📦 Downloading demo subset (~200MB)...
This may take a minute depending on your connection.
Retrieving folder contents
Processing file 1oBqN7qmFWADx8iM0NkAYd7aK-WAhql8y adv(64bit)_f30k_img.npy
Processing file 11EJBvb0WfN-m8T6dNFnleMAmxdeVVjzS adv(64bit)_f30k_txt.npy
Processing file 1yqZYYuYFzToIXRDAekJGK2lw2mfaUViS beit3_coco_img.npy
Processing file 1YX-sXIJBaiRSkEauCArbab4fZ8K-ZKtT beit3_coco_txt.npy
Processing file 1me4Y3EzCMMktPye_yjbclYAdMasQOKDk beit3_f30k_img.npy
Processing file 1fvibbzqdaYifHwUxUzs7_BfDOcT-X_Zc beit3_f30k_txt.npy
Processing file 1d5RVbFU88Zqftjp7szmuvXZxHRhzS5Nc blip2_f30k_sim_i2t.npy
Processing file 1uGUkED5Pjft05MEU0up-h6PoLiQ5VcTU blip2_f30k_sim_t2i.npy
Processing file 1WMkvmiz2ZKRlS1HqUeFyWZwH9DHJKTag coco-karpathy-testall-labels.npy
Processing file 1T2M2jZIxioFsHBgwtNWHag2SjHcz38D5 dadh_f30k_img.npy
Processing file 1EMkMKxuLan1RAh_P5pdoUyIemWxJ9J-l dadh_f30k_txt.npy
Processing file 1s2hG8nlWZ8IDALm6n-_hRIiq7bYQNgyG flickr30k-karpathy-

## 1. Core Concepts

FiCo-ITR evaluates two types of retrieval:
- **Instance-level**: Finding the exact matching caption/image
- **Category-level**: Finding captions/images with similar semantic categories

## 2. Case 1: Standard Format (Separate Embeddings)

Most models provide separate image and text embeddings. Here's **BEiT-3** on Flickr30k:

In [2]:
import numpy as np
from fico_itr import compute_similarity, instance_retrieval, category_retrieval

# Load embeddings
img_embs = np.load('results_data/beit3_f30k_img.npy')
txt_embs = np.load('results_data/beit3_f30k_txt.npy')

print(f"Image embeddings: {img_embs.shape}")  # (1000, 768)
print(f"Text embeddings: {txt_embs.shape}")   # (5000, 768)

Image embeddings: (1000, 768)
Text embeddings: (5000, 768)


In [3]:
# Step 1: Compute similarity matrix
similarity_matrix = compute_similarity(img_embs, txt_embs, measure='cosine')
print(f"Similarity matrix: {similarity_matrix.shape}")  # (1000, 5000)

Similarity matrix: (1000, 5000)


In [4]:
# Step 2: Instance-level retrieval
i2t_results, t2i_results = instance_retrieval(similarity_matrix)

print(f"BEiT-3 Flickr30k Results:")
print(f"Image→Text: R@1={i2t_results['R@1']:.1f}% R@5={i2t_results['R@5']:.1f}% R@10={i2t_results['R@10']:.1f}%")
print(f"Text→Image: R@1={t2i_results['R@1']:.1f}% R@5={t2i_results['R@5']:.1f}% R@10={t2i_results['R@10']:.1f}%")

BEiT-3 Flickr30k Results:
Image→Text: R@1=96.3% R@5=99.7% R@10=100.0%
Text→Image: R@1=86.2% R@5=97.7% R@10=98.8%


In [5]:
# Step 3: Category-level retrieval
labels = np.load('results_data/flickr30k-karpathy-test-labels.npy')
i2t_map, t2i_map = category_retrieval(similarity_matrix, labels)

print(f"I2T mAP: {i2t_map:.3f}  |  T2I mAP: {t2i_map:.3f}")

I2T mAP: 0.944  |  T2I mAP: 0.948


## 3. Case 2: Pre-computed Similarity Matrix

Some models provide similarity matrices directly. Here's **SCAN**:

In [6]:
import numpy as np
from fico_itr import instance_retrieval

# SCAN provides pre-computed similarity matrix
similarity_matrix = np.load('results_data/scan_f30k_sim.npy')
print(f"Pre-computed similarity: {similarity_matrix.shape}")  # (1000, 5000)

# Evaluate directly - no need to compute similarity
i2t_results, t2i_results = instance_retrieval(similarity_matrix)

print(f"\nSCAN Flickr30k Results:")
print(f"Image→Text: R@1={i2t_results['R@1']:.1f}% R@5={i2t_results['R@5']:.1f}% R@10={i2t_results['R@10']:.1f}%")
print(f"Text→Image: R@1={t2i_results['R@1']:.1f}% R@5={t2i_results['R@5']:.1f}% R@10={t2i_results['R@10']:.1f}%")

Pre-computed similarity: (1000, 5000)

SCAN Flickr30k Results:
Image→Text: R@1=66.7% R@5=89.3% R@10=94.0%
Text→Image: R@1=43.1% R@5=73.4% R@10=82.2%


## 4. Case 3: Separate Directional Matrices

**BLIP-2** uses different similarity matrices for each retrieval direction:

In [7]:
import numpy as np
from fico_itr import instance_retrieval

# BLIP-2 has task-specific fine-tuned matrices
sim_i2t = np.load('results_data/blip2_f30k_sim_i2t.npy')  # For image→text
sim_t2i = np.load('results_data/blip2_f30k_sim_t2i.npy')  # For text→image

print(f"I2T matrix: {sim_i2t.shape}")
print(f"T2I matrix: {sim_t2i.shape}")

# Pass both matrices
i2t_results, t2i_results = instance_retrieval(sim_i2t, sim_t2i)

print(f"\nBLIP-2 Flickr30k Results:")
print(f"Image→Text: R@1={i2t_results['R@1']:.1f}% R@5={i2t_results['R@5']:.1f}% R@10={i2t_results['R@10']:.1f}%")
print(f"Text→Image: R@1={t2i_results['R@1']:.1f}% R@5={t2i_results['R@5']:.1f}% R@10={t2i_results['R@10']:.1f}%")

I2T matrix: (1000, 5000)
T2I matrix: (5000, 1000)


/usr/local/lib/python3.11/dist-packages/fico_itr/tasks.py:337: UserWarning: Transposing similarity matrix from (5000, 1000) to (1000, 5000) (expected images × captions)
  warnings.warn(



BLIP-2 Flickr30k Results:
Image→Text: R@1=97.6% R@5=100.0% R@10=100.0%
Text→Image: R@1=89.7% R@5=98.2% R@10=98.9%


## 5. Case 4: Non-uniform Caption Distribution

MS-COCO has variable numbers of captions per image. **BEiT-3** on COCO:

In [8]:
import numpy as np
from fico_itr import compute_similarity, instance_retrieval

# BEiT-3 on COCO with 25010 captions (non-uniform distribution)
img_embs = np.load('results_data/beit3_coco_img.npy')
txt_embs = np.load('results_data/beit3_coco_txt.npy')

# Load caption-to-image mapping
caption_mapping = np.load('results_data/mscoco_test_indices.npy').tolist()

print(f"Images: {img_embs.shape[0]}, Captions: {txt_embs.shape[0]}")
print(f"Caption distribution: {min(caption_mapping.count(i) for i in set(caption_mapping))}-{max(caption_mapping.count(i) for i in set(caption_mapping))} per image")

# Evaluate with caption mapping
similarity_matrix = compute_similarity(img_embs, txt_embs)
i2t_results, t2i_results = instance_retrieval(similarity_matrix, captions_per_image=caption_mapping)

print(f"\nBEiT-3 COCO Results:")
print(f"Image→Text: R@1={i2t_results['R@1']:.1f}% R@5={i2t_results['R@5']:.1f}% R@10={i2t_results['R@10']:.1f}%")
print(f"Text→Image: R@1={t2i_results['R@1']:.1f}% R@5={t2i_results['R@5']:.1f}% R@10={t2i_results['R@10']:.1f}%")

Images: 5000, Captions: 25010
Caption distribution: 5-6 per image

BEiT-3 COCO Results:
Image→Text: R@1=79.0% R@5=94.4% R@10=97.2%
Text→Image: R@1=61.4% R@5=84.6% R@10=90.7%


## 6. Complete Evaluation

Now let's run all models from our demo. This demonstrates FiCo-ITR's ability to handle all formats automatically. Make sure to manually download all embeddings in the first block, as the automatic "demo" option only includes a few models

In [10]:
#@title Select dataset and run complete evaluation { display-mode: "form" }
dataset = "coco" #@param ["f30k", "coco"]

# All necessary imports
import numpy as np
import glob
from fico_itr import compute_similarity, instance_retrieval

# Helper functions
def load_data(model, dataset, modality=None):
    if modality:
        return np.load(f'results_data/{model}_{dataset}_{modality}.npy')
    else:
        return np.load(f'results_data/{model}_{dataset}_sim.npy')

def is_precomputed(model):
    return model in ['blip2', 'xvlm', 'imram', 'scan'] or model.endswith('(nf)')

# Models available in demo subset
demo_models = ["beit3", "scan", "blip2", "dadh", "adv(64bit)", "ucch"]

# Check which models are available
available_files = set(glob.glob('results_data/*.npy'))
available_models = []

for model in demo_models:
    # Check if model has files for selected dataset
    if dataset == "coco" and model not in ["beit3"]:
        continue  # Only beit3 has COCO in demo
    if any(f"{model}_{dataset}" in f for f in available_files):
        available_models.append(model)

if not available_models:
    print(f"⚠️ No models found for {dataset}. Make sure data is downloaded.")
else:
    # Load labels
    labels = np.load(f'results_data/{"flickr30k" if dataset == "f30k" else "coco"}-karpathy-{"test" if dataset == "f30k" else "testall"}-labels.npy')

    print(f"Evaluating {len(available_models)} models on {dataset.upper()}")
    print("=" * 80)

    results = {}
    for model in available_models:
        try:
            # Handle caption distribution
            if dataset == 'coco' and model == 'beit3':
                captions_per_image = np.load('results_data/mscoco_test_indices.npy').tolist()
            elif model in ['vsrn', 'ucch']:  # Square matrices
                captions_per_image = 5
            else:
                captions_per_image = 5

            # Load data based on format
            if model in ['blip2', 'xvlm']:  # Separate directional matrices
                sim_i2t = load_data(model, dataset, 'sim_i2t')
                sim_t2i = load_data(model, dataset, 'sim_t2i')
                i2t, t2i = instance_retrieval(sim_i2t, sim_t2i, captions_per_image=captions_per_image)
            elif is_precomputed(model):  # Pre-computed similarity
                sim = load_data(model, dataset)
                i2t, t2i = instance_retrieval(sim, captions_per_image=captions_per_image)
            else:  # Separate embeddings
                img_embs = load_data(model, dataset, 'img')
                txt_embs = load_data(model, dataset, 'txt')
                sim = compute_similarity(img_embs, txt_embs)
                i2t, t2i = instance_retrieval(sim, captions_per_image=captions_per_image)

            results[model] = {
                'i2t': {'R@1': i2t['R@1'], 'R@5': i2t['R@5'], 'R@10': i2t['R@10']},
                't2i': {'R@1': t2i['R@1'], 'R@5': t2i['R@5'], 'R@10': t2i['R@10']}
            }
            print(f"\n{model}:")
            print(f"  I→T: R@1={i2t['R@1']:>5.1f}% R@5={i2t['R@5']:>5.1f}% R@10={i2t['R@10']:>5.1f}%")
            print(f"  T→I: R@1={t2i['R@1']:>5.1f}% R@5={t2i['R@5']:>5.1f}% R@10={t2i['R@10']:>5.1f}%")

        except Exception as e:
            print(f"\n{model}: Error - {str(e)}")

    # Find best models
    if results:
        print("\n" + "=" * 80)
        print("🏆 Best performing:")

        # Best for each metric
        metrics = [('R@1', 'R@1'), ('R@5', 'R@5'), ('R@10', 'R@10')]
        for metric_name, metric_key in metrics:
            best_i2t = max(results.items(), key=lambda x: x[1]['i2t'][metric_key])
            best_t2i = max(results.items(), key=lambda x: x[1]['t2i'][metric_key])
            print(f"\n{metric_name}:")
            print(f"  I→T: {best_i2t[0]} ({best_i2t[1]['i2t'][metric_key]:.1f}%)")
            print(f"  T→I: {best_t2i[0]} ({best_t2i[1]['t2i'][metric_key]:.1f}%)")

Evaluating 1 models on COCO

beit3:
  I→T: R@1= 79.0% R@5= 94.4% R@10= 97.2%
  T→I: R@1= 61.4% R@5= 84.6% R@10= 90.7%

🏆 Best performing:

R@1:
  I→T: beit3 (79.0%)
  T→I: beit3 (61.4%)

R@5:
  I→T: beit3 (94.4%)
  T→I: beit3 (84.6%)

R@10:
  I→T: beit3 (97.2%)
  T→I: beit3 (90.7%)


## 7. Using Your Own Model

To evaluate your own model with FiCo-ITR:

```python
# Extract features from your model
img_features = your_model.encode_images(images)   # Shape: (n_images, d)
txt_features = your_model.encode_texts(captions)  # Shape: (n_captions, d)

# Evaluate with FiCo-ITR
similarity = compute_similarity(img_features, txt_features)
i2t_results, t2i_results = instance_retrieval(similarity)

print(f"Your model - I2T R@1: {i2t_results['R@1']:.1f}%")
```

## Resources

- 📚 [Documentation](https://github.com/MikelWL/FiCo-ITR/tree/main/docs) - Technical details
- 🛠️ [Toolkit](https://github.com/MikelWL/FiCo-ITR/tree/main/toolkit) - Feature extraction examples
- 📄 [Paper](https://link.springer.com/article/10.1007/s13735-025-00368-6) - Full methodology

If you find FiCo-ITR useful, please cite our paper!